

# **Laboratorio 11: Pienso, luego predigo 💡**

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2025</strong></center>

### Cuerpo Docente:

- Profesores: Diego Cortez, Gabriel Iturra
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Nicolás Cabello, Cristopher Urbina

### **Equipo:**

- Nombre de alumno 1: Naomí Cautivo B.
- Nombre de alumno 2: Máximo Flores Valenzuela

### **Link de repositorio de GitHub:** [maxfloresv/MDS7202](https://github.com/maxfloresv/MDS7202)

## **Temas a tratar**

- Reinforcement Learning
- Large Language Models

## **Reglas:**

- **Grupos de 2 personas**
- Fecha de entrega: Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda **fuertemente** asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

### **Objetivos principales del laboratorio**

- Resolución de problemas secuenciales usando Reinforcement Learning
- Habilitar un Chatbot para entregar respuestas útiles usando Large Language Models.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

## **1. Reinforcement Learning (2.0 puntos)**

En esta sección van a usar métodos de RL para resolver dos problemas interesantes: `Blackjack` y `LunarLander`.

In [1]:
%pip install -qqq gymnasium stable_baselines3
%pip install -qqq swig
%pip install -qqq gymnasium[box2d]

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### **1.1 Blackjack (1.0 puntos)**

<p align="center">
  <img src="https://www.recreoviral.com/wp-content/uploads/2016/08/s3.amazonaws.com-Math.gif"
" width="400">
</p>

La idea de esta subsección es que puedan implementar métodos de RL y así generar una estrategia para jugar el clásico juego Blackjack y de paso puedan ~~hacerse millonarios~~ aprender a resolver problemas mediante RL.

Comencemos primero preparando el ambiente. El siguiente bloque de código transforma las observaciones del ambiente a `np.array`:


In [2]:
import gymnasium as gym
from gymnasium.spaces import MultiDiscrete
import numpy as np

class FlattenObservation(gym.ObservationWrapper):
    def __init__(self, env):
        super(FlattenObservation, self).__init__(env)
        self.observation_space = MultiDiscrete(np.array([32, 11, 2]))

    def observation(self, observation):
        return np.array(observation).flatten()

env = gym.make("Blackjack-v1")
env = FlattenObservation(env)

#### **1.1.1 Descripción de MDP (0.2 puntos)**

Entregue una breve descripción sobre el ambiente [Blackjack](https://gymnasium.farama.org/environments/toy_text/blackjack/) y su formulación en MDP, distinguiendo de forma clara y concisa los estados, acciones y recompensas.

> **Respuesta**: El ambiente Blackjack es una simulación del juego de casino que lleva el mismo nombre, donde el objetivo del jugador es pedir cartas intentando no pasarse de $21$ ($\textsf{J}$, $\textsf{Q}$ y $\textsf{K}$ cuentan como $10$; el $\textsf{As}$ puede contar como $1$ o como $11$ ($\textsf{As}$ usable: siempre lo es si no hace que el jugador se exceda de $21$), y los números del $2$ al $10$ tienen el mismo valor que indica la carta), pero quedando lo más cerca de este número por debajo.
>
> El jugador pierde si se excede de $21$, o el _dealer_ (que inicialmente tiene una carta boca arriba y una boca abajo) obtiene un número más cercano por debajo de $21$ que el obtenido, sacando cartas hasta que la suma de su mano sea mayor o igual que $17$.
>
> - Los **estados** $S$ corresponden a una $3$-tupla de la forma $(s, d, u_\textsf{As})$, donde $s \in \mathcal{U}_s \cup \{4, 5, \dots, 31\}$ es la suma de la mano del jugador ($\mathcal{U}_s = \{0, 1, 2, 3\}$ es el conjunto de valores inalcanzables, pero se definen por rigurosidad matemática dentro de los valores de $s$, porque según la documentación, la primera coordenada puede tomar $32$ valores posibles), $d \in \mathcal{U}_d \cup \{1, \dots, 10\}$ es el valor de la carta que muestra el _dealer_ ($\mathcal{U}_d = \{0\}$ se agrega por el mismo motivo ya mencionado), y $u_\textsf{As} \in \{0, 1\}$ es un valor _booleano_ que indica si el $\textsf{As}$ tiene el valor de $11$ o no. 
>
>   El conjunto de estados iniciales $S_0 = \{s \in S \mid s \text{ es estado inicial}\}$ puede contener cualquier $3$-tupla de la misma forma donde $s \in \{4, 5, \dots, 21\}$, $d \in \{1, 2, \dots, 10\}$ y $u_\textsf{As} \in \{0, 1\}$.
> - Las **acciones** $A$ son dos, codificadas como $0$ y $1$ ($A = \{0,1\}$), que representan "dejar de pedir cartas" y "seguir pidiendo" respectivamente.
> - Las **recompensas** $R$ son cuatro: $+1$ si el jugador gana el juego (con o sin _blackjack_), $-1$ si pierde, $0$ si empata, $+1.5$ si gana con un _blackjack_ natural (es decir, que su mano empiece con un $10$ y un $\textsf{As}$ usable). Esta última es opcional, y se puede controlar con el hiperparámetro `natural: bool`.
>
> La formulación del problema como un MDP se puede realizar con estos conjuntos definidos anteriormente. Usualmente, toma la forma de una $4$-tupla $(S_t, A_t, R_{t+1}, S_{t+1})$, también llamada "experiencia".

#### **1.1.2 Generando un Baseline (0.2 puntos)**

Simule un escenario en donde se escojan acciones aleatorias. Repita esta simulación 5000 veces y reporte el promedio y desviación de las recompensas. ¿Cómo calificaría el performance de esta política? ¿Cómo podría interpretar las recompensas obtenidas?

> _Observación_: A pesar de que establecí la semilla aleatoria como $42$ (estándar científico), no pude obtener resultados consistentes en todas las ejecuciones. Por este motivo, los resultados que se enuncian a continuación son representativos de varias ejecuciones, pero pueden no ser completamente replicables.

> **Respuesta**: 
> - **¿Cómo calificaría el performance de esta política?** El _performance_ de esta política no es óptimo, pero tampoco es el peor, dado que la recompensa final en promedio es muy cercana a $0$ ($\hat{\mu} = 0\text{,}01$), lo que indica que una estrategia aleatoria tenderá a hacernos empatar. Lo que es negativo es la incertidumbre, $\hat{\sigma} = 1$, que es muy alta para el dominio de $R$, porque no nos asegura ninguna certeza de qué resultado obtendremos con esta estrategia. Esto ocurre naturalmente porque le estamos dejando el destino al azar.
> - **¿Cómo podría interpretar las recompensas obtenidas?** Las recompensas obtenidas por experiencia sólo pueden tomar una cantidad discreta de valores en $R=\{-1, 0, 1\}$ ($R' = R \cup \{1\text{,}5\}$ si se añade el _blackjack_ natural) según se pierda, empate o gane respectivamente. Por ejemplo, si una experiencia arroja valor $-1$, quiere decir que en ese juego, el jugador perdió. Matemáticamente, tiene sentido definir $R_t = 0$ en un instante $t$ que corresponda a un paso intermedio, dado que no termina el juego, y por ende, no podemos saber el resultado.

In [3]:
import numpy as np

MAX_EPISODES = 5000
SEED = 42
np.random.seed(SEED)

rewards = []

for episode in range(MAX_EPISODES):
  # Reset to initial state
  obs, info = env.reset(seed=SEED)
  done = False
  episode_reward = 0

  while not done:
    # Choose random action
    action = env.action_space.sample()
    new_obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    episode_reward += reward
    obs = new_obs

  rewards.append(episode_reward)

  if episode % 1000 == 0:
    print(f"\nEpisode {episode} finished.")
    print(f"Total reward: {episode_reward}.")

env.close()

mean = np.mean(rewards)
# Unbiased estimator
std = np.std(rewards, ddof=1)

print(f"\nRewards mean: {mean:.3f}")
print(f"Rewards STD: {std:.3f}")


Episode 0 finished.
Total reward: 1.0.

Episode 1000 finished.
Total reward: -1.0.

Episode 2000 finished.
Total reward: 1.0.

Episode 3000 finished.
Total reward: 1.0.

Episode 4000 finished.
Total reward: -1.0.

Rewards mean: 0.010
Rewards STD: 1.000


#### **1.1.3 Entrenamiento de modelo (0.2 puntos)**

A partir del siguiente [enlace](https://stable-baselines3.readthedocs.io/en/master/guide/algos.html), escoja un modelo de `stable_baselines3` y entrenelo para resolver el ambiente `Blackjack`.

In [4]:
import os
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

os.makedirs('models', exist_ok=True)
SAVE_PATH = "models/ppo_blackjack.zip"

if not os.path.exists(SAVE_PATH):
  # Vectorization is necessary to execute PPO
  # Used n_envs=1 to represent a DummyVecEnv (without paralelization)
  env = make_vec_env(lambda: env, n_envs=1, seed=SEED) 

  # MlpPolicy is for discrete environments
  model = PPO("MlpPolicy", env, seed=SEED)

  model.learn(total_timesteps=5e5)
  model.save(SAVE_PATH)
else:
  model = PPO.load(SAVE_PATH)

env.close()

#### **1.1.4 Evaluación de modelo (0.2 puntos)**

Repita el ejercicio 1.1.2 pero utilizando el modelo entrenado. ¿Cómo es el performance de su agente? ¿Es mejor o peor que el escenario baseline?

> **Respuesta**:
> - **¿Cómo es el performance de su agente?** **¿Es mejor o peor que el escenario baseline?** El _performance_ de este nuevo agente es ligeramente peor que el del modelo _baseline_, dado que el valor promedio de las recompensas baja del umbral $0$ ($\hat{\mu}_\text{PPO} = -0\text{,}05$), lo que implica que la tendencia ahora está favoreciendo el escenario donde el jugador pierde. Además, la nueva estimación de la desviación estándar ($\hat{\sigma}_\text{PPO} = 0\text{,}95$) tampoco es significativamente distinta a la del modelo _baseline_, entonces aplica el mismo argumento de incerteza.

In [5]:
from stable_baselines3.common.evaluation import evaluate_policy

N_EVAL_EPISODES = 5000

env = gym.make("Blackjack-v1")
env = FlattenObservation(env)

# Unique seed for eval environment
eval_env = make_vec_env(lambda: env, n_envs=1, seed=2*SEED) 

mean_reward, std_reward = evaluate_policy(
  model, 
  eval_env, 
  n_eval_episodes=N_EVAL_EPISODES
)

eval_env.close()

print(f"Rewards mean: {mean_reward:.3f}")
print(f"Rewards STD: {std_reward:.3f}")

Rewards mean: -0.050
Rewards STD: 0.950


#### **1.1.5 Estudio de acciones (0.2 puntos)**

Genere una función que reciba un estado y retorne la accion del agente. Luego, use esta función para entregar la acción escogida frente a los siguientes escenarios:

- Suma de cartas del agente es 6, dealer muestra un 7, agente no tiene tiene un as
- Suma de cartas del agente es 19, dealer muestra un 3, agente tiene tiene un as

¿Son coherentes sus acciones con las reglas del juego?

> **Respuesta**: La acción tomada para el primer escenario $(s, d, u_\textsf{As}) = (6, 7, 0)$ es `Hit`, es decir, el modelo prefiere que el jugador siga sacando cartas. Esto tiene sentido, porque plantarse con $6$ haría que el jugador inmediatamente pierda ($6 < 7$). En el segundo escenario, la decisión tomada es `Stick`. Esto también es coherente con las reglas del juego, porque teniendo $19$ en mano el jugador y el _dealer_ $3$, es muy poco probable que este último supere los $19$ sin pasarse.

Hint: ¿A que clase de python pertenecen los estados? Pruebe a usar el método `.reset` para saberlo.

In [8]:
def get_agent_action(model: PPO, observation: np.ndarray) -> int:
  """
  Gets the agent action based on a trained model (in this case, PPO).
  """
  action, _ = model.predict(observation, deterministic=True)
  return action

scenario_1 = np.array([6, 7, 0])
action_1 = get_agent_action(model, scenario_1)

print(f"Scenario 1 - Action: {'Stick' if action_1 == 0 else 'Hit'}")

scenario_2 = np.array([19, 3, 1])
action_2 = get_agent_action(model, scenario_2)

print(f"Scenario 2 - Action: {'Stick' if action_2 == 0 else 'Hit'}")

Scenario 1 - Action: Hit
Scenario 2 - Action: Stick


### **1.2 LunarLander**

<p align="center">
  <img src="https://i.redd.it/097t6tk29zf51.jpg"
" width="400">
</p>

Similar a la sección 2.1, en esta sección usted se encargará de implementar una gente de RL que pueda resolver el ambiente `LunarLander`.

Comencemos preparando el ambiente:


In [ ]:
import gymnasium as gym
env = gym.make("LunarLander-v2", render_mode = "rgb_array", continuous = True) # notar el parámetro continuous = True

Noten que se especifica el parámetro `continuous = True`. ¿Que implicancias tiene esto sobre el ambiente?

Además, se le facilita la función `export_gif` para el ejercicio 2.2.4:

In [ ]:
import imageio
import numpy as np

def export_gif(model, n = 5):
  '''
  función que exporta a gif el comportamiento del agente en n episodios
  '''
  images = []
  for episode in range(n):
    obs = model.env.reset()
    img = model.env.render()
    done = False
    while not done:
      images.append(img)
      action, _ = model.predict(obs)
      obs, reward, done, info = model.env.step(action)
      img = model.env.render(mode="rgb_array")

  imageio.mimsave("agent_performance.gif", [np.array(img) for i, img in enumerate(images) if i%2 == 0], fps=29)

#### **1.2.1 Descripción de MDP (0.2 puntos)**

Entregue una breve descripción sobre el ambiente [LunarLander](https://gymnasium.farama.org/environments/box2d/lunar_lander/) y su formulación en MDP, distinguiendo de forma clara y concisa los estados, acciones y recompensas. ¿Como se distinguen las acciones de este ambiente en comparación a `Blackjack`?

Nota: recuerde que se especificó el parámetro `continuous = True`

`escriba su respuesta acá`

#### **1.2.2 Generando un Baseline (0.2 puntos)**

Simule un escenario en donde se escojan acciones aleatorias. Repita esta simulación 10 veces y reporte el promedio y desviación de las recompensas. ¿Cómo calificaría el performance de esta política?

#### **1.2.3 Entrenamiento de modelo (0.2 puntos)**

A partir del siguiente [enlace](https://stable-baselines3.readthedocs.io/en/master/guide/algos.html), escoja un modelo de `stable_baselines3` y entrenelo para resolver el ambiente `LunarLander` **usando 10000 timesteps de entrenamiento**.

#### **1.2.4 Evaluación de modelo (0.2 puntos)**

Repita el ejercicio 1.2.2 pero utilizando el modelo entrenado. ¿Cómo es el performance de su agente? ¿Es mejor o peor que el escenario baseline?

#### **1.2.5 Optimización de modelo (0.2 puntos)**

Repita los ejercicios 1.2.3 y 1.2.4 hasta obtener un nivel de recompensas promedio mayor a 50. Para esto, puede cambiar manualmente parámetros como:
- `total_timesteps`
- `learning_rate`
- `batch_size`

Una vez optimizado el modelo, use la función `export_gif` para estudiar el comportamiento de su agente en la resolución del ambiente y comente sobre sus resultados.

Adjunte el gif generado en su entrega (mejor aún si además adjuntan el gif en el markdown).

## **2. Large Language Models (4.0 puntos)**

En esta sección se enfocarán en habilitar un Chatbot que nos permita responder preguntas útiles a través de LLMs.

### **2.0 Configuración Inicial**

<p align="center">
  <img src="https://media1.tenor.com/m/uqAs9atZH58AAAAd/config-config-issue.gif"
" width="400">
</p>

Como siempre, cargamos todas nuestras API KEY al entorno:

In [ ]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

### **2.1 Retrieval Augmented Generation (1.5 puntos)**

<p align="center">
  <img src="https://y.yarn.co/218aaa02-c47e-4ec9-b1c9-07792a06a88f_text.gif"
" width="400">
</p>

El objetivo de esta subsección es que habiliten un chatbot que pueda responder preguntas usando información contenida en documentos PDF a través de **Retrieval Augmented Generation.**

#### **2.1.1 Reunir Documentos (0 puntos)**

Reuna documentos PDF sobre los que hacer preguntas siguiendo las siguientes instrucciones:
  - 2 documentos .pdf como mínimo.
  - 50 páginas de contenido como mínimo entre todos los documentos.
  - Ideas para documentos: Documentos relacionados a temas académicos, laborales o de ocio. Aprovechen este ejercicio para construir algo útil y/o relevante para ustedes!
  - Deben ocupar documentos reales, no pueden utilizar los mismos de la clase.
  - Deben registrar sus documentos en la siguiente [planilla](https://docs.google.com/spreadsheets/d/1Hy1w_dOiG2UCHJ8muyxhdKPZEPrrL7BNHm6E90imIIM/edit?usp=sharing). **NO PUEDEN USAR LOS MISMOS DOCUMENTOS QUE OTRO GRUPO**
  - **Recuerden adjuntar los documentos en su entrega**.

In [ ]:
%pip install --upgrade --quiet PyPDF2

In [ ]:
import PyPDF2

doc_paths = [] # rellenar con los path a sus documentos

assert len(doc_paths) >= 2, "Deben adjuntar un mínimo de 2 documentos"

total_paginas = sum(len(PyPDF2.PdfReader(open(doc, "rb")).pages) for doc in doc_paths)
assert total_paginas >= 50, f"Páginas insuficientes: {total_paginas}"

#### **2.1.2 Vectorizar Documentos (0.2 puntos)**

Vectorice los documentos y almacene sus representaciones de manera acorde.

#### **2.1.3 Habilitar RAG (0.3 puntos)**

Habilite la solución RAG a través de una *chain* y guárdela en una variable.

#### **2.1.4 Verificación de respuestas (0.5 puntos)**

Genere un listado de 3 tuplas ("pregunta", "respuesta correcta") y analice la respuesta de su solución para cada una. ¿Su solución RAG entrega las respuestas que esperaba?

Ejemplo de tupla:
- Pregunta: ¿Quién es el presidente de Chile?
- Respuesta correcta: El presidente de Chile es Gabriel Boric

#### **2.1.5 Sensibilidad de Hiperparámetros (0.5 puntos)**

Extienda el análisis del punto 2.1.4 analizando cómo cambian las respuestas entregadas cambiando los siguientes hiperparámetros:
- `Tamaño del chunk`. (*¿Cómo repercute que los chunks sean mas grandes o chicos?*)
- `La cantidad de chunks recuperados`. (*¿Qué pasa si se devuelven muchos/pocos chunks?*)
- `El tipo de búsqueda`. (*¿Cómo afecta el tipo de búsqueda a las respuestas de mi RAG?*)

### **2.2 Agentes (1.0 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/rcqnN2aJCSEAAAAd/secret-agent-man.gif"
" width="400">
</p>

Similar a la sección anterior, en esta sección se busca habilitar **Agentes** para obtener información a través de tools y así responder la pregunta del usuario.

#### **2.2.1 Tool de Tavily (0.2 puntos)**

Generar una *tool* que pueda hacer consultas al motor de búsqueda **Tavily**.

#### **2.2.2 Tool de Wikipedia (0.2 puntos)**

Generar una *tool* que pueda hacer consultas a **Wikipedia**.

*Hint: Le puede ser de ayuda el siguiente [link](https://python.langchain.com/v0.1/docs/modules/tools/).*

#### **2.2.3 Crear Agente (0.3 puntos)**

Crear un agente que pueda responder preguntas preguntas usando las *tools* antes generadas. Asegúrese que su agente responda en español. Por último, guarde el agente en una variable.

#### **2.2.4 Verificación de respuestas (0.3 puntos)**

Pruebe el funcionamiento de su agente y asegúrese que el agente esté ocupando correctamente las tools disponibles. ¿En qué casos el agente debería ocupar la tool de Tavily? ¿En qué casos debería ocupar la tool de Wikipedia?

### **2.3 Multi Agente (1.5 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/r7QMJLxU4BoAAAAd/this-is-getting-out-of-hand-star-wars.gif"
" width="450">
</p>

El objetivo de esta subsección es encapsular las funcionalidades creadas en una solución multiagente con un **supervisor**.


#### **2.3.1 Generando Tools (0.5 puntos)**

Transforme la solución RAG de la sección 2.1 y el agente de la sección 2.2 a *tools* (una tool por cada uno).

#### **2.3.2 Agente Supervisor (0.5 puntos)**

Habilite un agente que tenga acceso a las tools del punto anterior y pueda responder preguntas relacionadas. Almacene este agente en una variable llamada supervisor.

#### **2.3.3 Verificación de respuestas (0.25 puntos)**

Pruebe el funcionamiento de su agente repitiendo las preguntas realizadas en las secciones 2.1.4 y 2.2.4 y comente sus resultados. ¿Cómo varían las respuestas bajo este enfoque?

#### **2.3.4 Análisis (0.25 puntos)**

¿Qué diferencias tiene este enfoque con la solución *Router* vista en clases? Nombre al menos una ventaja y desventaja.

`escriba su respuesta acá`

### **2.4 Memoria (Bonus +0.5 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/Gs95aiElrscAAAAd/memory-unlocked-ratatouille-critic.gif"
" width="400">
</p>

Una de las principales falencias de las soluciones que hemos visto hasta ahora es que nuestro chat no responde las interacciones anteriores, por ejemplo:

- Pregunta 1: "Hola! mi nombre es Sebastián"
  - Respuesta esperada: "Hola Sebastián! ..."
- Pregunta 2: "Cual es mi nombre?"
  - Respuesta actual: "Lo siento pero no conozco tu nombre :("
  - **Respuesta esperada: "Tu nombre es Sebastián"**

Para solucionar esto, se les solicita agregar un componente de **memoria** a la solución entregada en el punto 2.3.

**Nota: El Bonus es válido <u>sólo para la sección 2 de Large Language Models.</u>**

### **2.5 Despliegue (0 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/IytHqOp52EsAAAAd/you-get-a-deploy-deploy.gif"
" width="400">
</p>

Una vez tengan los puntos anteriores finalizados, toca la etapa de dar a conocer lo que hicimos! Para eso, vamos a desplegar nuestro modelo a través de `gradio`, una librería especializada en el levantamiento rápido de demos basadas en ML.

Primero instalamos la librería:

In [ ]:
%pip install --upgrade --quiet gradio

Luego sólo deben ejecutar el siguiente código e interactuar con la interfaz a través del notebook o del link generado:

In [ ]:
import gradio as gr
import time

def agent_response(message, history):
  '''
  Función para gradio, recibe mensaje e historial, devuelte la respuesta del chatbot.
  '''
  # get chatbot response
  response = ... # rellenar con la respuesta de su chat

  # assert
  assert type(response) == str, "output de route_question debe ser string"

  # "streaming" response
  for i in range(len(response)):
    time.sleep(0.015)
    yield response[: i+1]

gr.ChatInterface(
    agent_response,
    type="messages",
    title="Chatbot MDS7202", # Pueden cambiar esto si lo desean
    description="Hola! Soy un chatbot muy útil :)", # también la descripción
    theme="soft",
    ).launch(
        share=True, # pueden compartir el link a sus amig@s para que interactuen con su chat!
        debug = False,
        )

# Conclusión
Éxito!
<center>
<img src ="https://media.tenor.com/MRQgxcelAV8AAAAM/perry-the-platypus-phineas-and-ferb.gif" width = 400 />